# Feature Engineering II: Column Transformers and Pipelines
<img src = "./assets/lego.webp" width = "450">

<a href = "https://www.highsnobiety.com/p/lego-transformers-optimus-prime/">Image Source</a>

This notebook builds on the [feature engineering introduction notebook](1.1_intro_to_fe.ipynb) to automate the transformation process, simplifying our workflow and unlocking the potential of the sklearn library.
<hr style="border:2px solid black">

## Penguin Dataset

We will use the Palmer Penguin Dataset.

### Business Goal
> Predict the penguin body mass given the input feature : flipper_length_mm, bill_length_mm, species and sex

#### Load Packages

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

#### Load Data

In [ ]:
df = pd.read_csv("./data/penguins.csv")
df.head()

#### Features and Target

In [ ]:
numerical_features = ["flipper_length_mm", "bill_length_mm"]

categorical_features = ["species", "sex"]

features = numerical_features + categorical_features

target_variable = "body_mass_g"

#### Feature-Target separation

In [ ]:
# Feature matrix
X = df[features]

# Target column
y = df[target_variable]

#### Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=88, shuffle=True, stratify=X["species"]
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

## Exploratory Data Analysis
Check issues with data:
+ which variable has missing values?
+ which variables are binary, categorical, metric?
+ do categorical variables have non-numeric values?
+ do metric features vary on different scales?


In [ ]:
# Assuming X_train is a DataFrame and y_train is a Series
df_train = pd.concat([X_train, y_train], axis=1)

print("Combined train data shape:", df_train.shape)

In [ ]:
df_train.isna().sum()

<hr style="border:2px solid black">

## Feature Engineering
We have a pair of tools, `ColumnTransformer()` and `Pipeline()`, which can dramatically simplify and automate feature engineering.

The complete model below follows this structure. Each branch receives its selected columns; its own steps run in sequence.

<img src="assets/feature_engineering_pipeline.svg" width="650" alt="Feature engineering pipeline: numeric and categorical branches join before linear regression.">

During `fit`, preprocessing is learned before the model is fitted. During `predict`, the same fitted preprocessing transforms the new inputs; it is not relearned.


### ColumnTransformer()
<a href="https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html">`ColumnTransformer` </a> allows us to specify which columns receive which transformations (and conveniently reintegrates the dataset).

Parameters:
 * `transformers` - list of tuples `(name, transformer, columns)`

 * `remainder` - controls untouched columns. Choose either `drop` (default) or `passthrough`<br></br>
  
>**Note** that `ColumnTransformer()` applies its branches independently, not as a sequence on the same data, so if a column is transformed more than once, the version generated by each of these transformations will be included.

Parallel computation is optional via `n_jobs`. To impute and then scale the same columns, put those steps in a `Pipeline` within one branch.

#### Building our first transformer


In [ ]:
transformers = [
    ("ohe", OneHotEncoder(drop="first", sparse_output=False), ["species", "sex"]),
    ("numeric_scaler", RobustScaler(), ["bill_length_mm", "flipper_length_mm"]),
]

In [ ]:
column_transformer = ColumnTransformer(transformers, remainder="drop")
column_transformer.fit_transform(X_train)

The first transformer leaves missing numerical values in place. The encoder can treat a missing category as a category of its own, but it does not impute it. Adding a separate imputation branch would create extra columns rather than feed imputed values into scaling or encoding. To impute first and then scale or encode the same columns, the operations must run in sequence.

What we need here is a way to sequentially apply transformations, which leads us nicely into...

### Pipeline()

<a href="https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html">`Pipeline()`</a> applies steps sequentially: each step receives the output of the previous step. It can transform an entire feature matrix; the column selection in our example is handled by `ColumnTransformer`. The final step can also be a predictive model.


Parameters:
 * `steps` - list of tuples `(name, transformer)`

#### Build a pipeline and integrate it into our transformer

In [ ]:
# Let's define the steps to impute and transform sex
sex_steps = [
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("sex_ohe", OneHotEncoder(drop="first", sparse_output=False)),
]

In [ ]:
# Let's instantiate the sex pipeline
sex_pipeline = Pipeline(steps=sex_steps)
sex_pipeline

In [ ]:
numeric_steps = [
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
]

In [ ]:
numeric_pipeline = Pipeline(steps=numeric_steps)
numeric_pipeline

In [ ]:
transformers_2 = [
    ("sex_pipeline", sex_pipeline, ["sex"]),
    ("species_ohe", OneHotEncoder(drop="first", sparse_output=False), ["species"]),
    ("numeric_pipeline", numeric_pipeline, numerical_features),
]

column_transformer_2 = ColumnTransformer(
    transformers=transformers_2, remainder="drop"
).set_output(transform="pandas")

#### Let's Try It Out!

In [ ]:
# Fit the column transformer object ONLY using train data
column_transformer_2.fit(X_train)

In [ ]:
X_train.isna().sum()

In [ ]:
# Transform the data
X_train_fe = column_transformer_2.transform(X_train)
X_train_fe

As you can see the variables have been transformed according to the strategies defined in our pipeline.

Both numerical columns use the same impute-then-scale pipeline. The output retains the original row indices.

## Model Building

### Nesting Pipelines

Pipeline steps must follow scikit-learn's estimator interface: intermediate steps provide `fit` and `transform`, while the final step can be a predictor. Wrap a custom function in `FunctionTransformer` rather than passing an arbitrary function directly. The final trick we'll explore with pipelines is the ability to nest several layers within one.

In [ ]:
# build a pipeline containing our complete transformer and then a linear regression model
model_steps = [
    ("feature_engineering", column_transformer_2),
    ("linear_regression", LinearRegression()),
]
linear_model = Pipeline(steps=model_steps)
linear_model

### Optional: Does More Complexity Help?

Before looking at test performance, compare two Ridge regression pipelines on **training data only**: the original numerical features versus degree-2 numerical features. Ridge is linear regression with a penalty on large coefficients; `alpha` controls that penalty. Both use the same regularization strength and the same five folds, so this is a small, controlled comparison, not an exhaustive search. Each fold fits its own imputation and scaling. `clone` creates an unfitted copy, leaving the main linear-regression demonstration below unchanged.

In [ ]:
from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import PolynomialFeatures

simple = clone(linear_model).set_params(linear_regression=Ridge(alpha=1.0))
expanded = clone(simple).set_params(
    feature_engineering__numeric_pipeline=Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
            ("scaler", RobustScaler()),
        ]
    )
)
folds = KFold(n_splits=5, shuffle=True, random_state=42)
comparison = []
for name, candidate in [("Original features", simple), ("Degree-2 features", expanded)]:
    # sklearn negates losses so higher scores are better; convert back to MAE.
    mae = -cross_val_score(
        candidate, X_train, y_train, cv=folds, scoring="neg_mean_absolute_error"
    )
    comparison.append(
        {"Pipeline": name, "CV MAE (g)": mae.mean(), "Fold SD (g)": mae.std()}
    )
pd.DataFrame(comparison).set_index("Pipeline").round(1)

Lower MAE is better. Fold standard deviation describes variation across these folds, not a confidence interval. Does the extra complexity reduce error enough to justify it? If results are similar, prefer the simpler pipeline. Do not assume polynomial features must win or use test scores to choose between candidates.

Continue with the predefined linear model below; this optional comparison does not replace it automatically.

**train model**

In [ ]:
linear_model.fit(X_train, y_train)

In [ ]:
training_score = linear_model.score(X_train, y_train)
print(f"training r2 score: {round(training_score, 6)}")

### Model Evaluation

**Model Weights**

Numeric coefficients apply to the imputed, robust-scaled features, not directly to millimeters: a one-unit increase is one training IQR. Category coefficients compare against the dropped reference category while other inputs stay fixed. The coefficients are associations, not causal effects, and their magnitudes alone do not rank feature importance.


In [ ]:
column_step = linear_model.steps[0][1]
column_step

In [ ]:
model_step = linear_model.steps[1][1]
model_step

In [ ]:
coef_model = pd.DataFrame(
    data=model_step.coef_.reshape(1, -1),
    columns=column_step.get_feature_names_out(),
    index=pd.Index(["weight"]),
)

coef_model["intercept"] = model_step.intercept_
coef_model

**Model Prediction**

In [ ]:
y_pred_test = linear_model.predict(X_test)
y_pred_test

**Model Performance**

In [ ]:
test_score = linear_model.score(X_test, y_test)
test_mae = mean_absolute_error(y_test, y_pred_test)
print(f"Test R²: {test_score:.3f}")
print(f"Test MAE: {test_mae:.0f} grams")

>By applying a pipeline to our test data, we ensure that the test data is treated in the exact same way as the data the model was trained on.

The fitted pipeline reuses training preprocessing parameters; it does not refit them on test data. R² is not an accuracy percentage: 1 is perfect, 0 matches the constant test-target-mean benchmark, and negative values are possible. MAE reports the average absolute error in grams.

This is one final evaluation of a predefined pipeline. Compare alternatives using training-set cross-validation, not repeated inspection of test scores.

<hr style="border:2px solid black">